# Лабораторная работа №4: Получение навыков создания системы для распознавания текста на изображениях с использованием нейронных сетей

## Roboflow

Скачиваем датасет, который сделали в Roboflow

In [ ]:
!pip install roboflow

  Using cached roboflow-1.3.7-py3-none-any.whl.metadata (11 kB)
  Using cached idna-3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached roboflow-1.3.7-py3-none-any.whl (195 kB)
Using cached idna-3.7-py3-none-any.whl (66 kB)
Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.9 MB)
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_KEY"))
project = rf.workspace("personal-workspace-9n1jn").project("object-detection-lab-uhkno")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Object-Detection-Lab-1 in yolov8:: 100%|██████████| 1049/1049 [00:00<00:00, 8200.76it/s]


## Модели YOLOv8n

### Загрузка обученной модели

Так как модель уже была обучена, то просто загрузим файл `best.pt` и будем использовать его

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving best.pt to best.pt


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")
print(model.info())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model summary: 130 layers, 3,011,043 parameters, 0 gradients, 8.2 GFLOPs
(130, 3011043, 0, 8.1941504)


### Обучение модели

Если модель надо обучить, то вот для этого код

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="road-traffic-yolov8",
    device=0
)

Скачиваем графики результатов обучения и саму модель

In [ ]:
from google.colab import files
files.download("/content/runs/detect/road-traffic-yolov8/results.png")
files.download("/content/runs/detect/road-traffic-yolov8/confusion_matrix.png")
files.download("/content/runs/detect/road-traffic-yolov8/weights/best.pt")

## Тест на тестовой выборке и метрики

Запуск валидации на тестовой выборке

In [24]:
metrics = model.val(
    data="/content/Object-Detection-Lab-1/data.yaml",
    split="test",
    imgsz=640,
    conf=0.3,
    iou=0.5,
    device=0
)

print(f"Precision:  {metrics.box.mp:.3f}")
print(f"Recall:     {metrics.box.mr:.3f}")
print(f"mAP@50:     {metrics.box.map50:.3f}")
print(f"mAP@50-95:  {metrics.box.map:.3f}")

Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1030.1±303.8 MB/s, size: 57.3 KB)
val: Scanning /content/Object-Detection-Lab-1/test/labels... 31 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 31/31 617.8it/s 0.1s
val: New cache created: /content/Object-Detection-Lab-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3s/it 2.5s
                   all         31        638       0.95      0.638      0.653      0.525
Speed: 5.7ms preprocess, 12.2ms inference, 0.0ms loss, 14.8ms postprocess per image
Results saved to /content/runs/detect/val
Precision:  0.950
Recall:     0.638
mAP@50:     0.653
mAP@50-95:  0.525


## Сегментация тестового изображения

In [25]:
import cv2
import os
import glob

# Берём первое изображение из тестовой выборки
test_images = glob.glob("/content/Object-Detection-Lab-1/test/images/*.jpg")
test_image_path = test_images[0]

# Запускаем детекцию
results = model(test_image_path, conf=0.3, device=0)

image = cv2.imread(test_image_path)
boxes = results[0].boxes.xyxy.cpu().numpy()

# Папка для сегментов
os.makedirs("/content/segments", exist_ok=True)

# Вырезаем каждую найденную машину в отдельный файл
for i, box in enumerate(boxes):
    x1, y1, x2, y2 = map(int, box)
    crop = image[y1:y2, x1:x2]
    save_path = f"/content/segments/car_{i+1}.jpg"
    cv2.imwrite(save_path, crop)
    print(f"Сохранён сегмент: {save_path}")

# Сохраняем исходное изображение с bbox
annotated = results[0].plot()
cv2.imwrite("/content/test_annotated.jpg", annotated)
print(f"\nВсего найдено машин: {len(boxes)}")
print("Аннотированное изображение сохранено: /content/test_annotated.jpg")


image 1/1 /content/Object-Detection-Lab-1/test/images/road-traffic_mp4-0198_jpg.rf.66725246ed56afde059c758faa3088fb.jpg: 640x640 31 cars, 10.0ms
Speed: 3.7ms preprocess, 10.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
Сохранён сегмент: /content/segments/car_1.jpg
Сохранён сегмент: /content/segments/car_2.jpg
Сохранён сегмент: /content/segments/car_3.jpg
Сохранён сегмент: /content/segments/car_4.jpg
Сохранён сегмент: /content/segments/car_5.jpg
Сохранён сегмент: /content/segments/car_6.jpg
Сохранён сегмент: /content/segments/car_7.jpg
Сохранён сегмент: /content/segments/car_8.jpg
Сохранён сегмент: /content/segments/car_9.jpg
Сохранён сегмент: /content/segments/car_10.jpg
Сохранён сегмент: /content/segments/car_11.jpg
Сохранён сегмент: /content/segments/car_12.jpg
Сохранён сегмент: /content/segments/car_13.jpg
Сохранён сегмент: /content/segments/car_14.jpg
Сохранён сегмент: /content/segments/car_15.jpg
Сохранён сегмент: /content/segments/car_16.jpg
Сохранён сегме

## Скачиваем изображения

In [26]:
from google.colab import files
files.download("/content/test_annotated.jpg")

# Скачиваем сегменты
import glob
for f in glob.glob("/content/segments/*.jpg"):
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Kaggle

Настройка Kaggle API. Задаём имя пользователя и API-ключ

In [ ]:
from google.colab import userdata
import json
import os

os.makedirs("/root/.kaggle", exist_ok=True)

creds = {
    "username": userdata.get("KAGGLE_USERNAME"),
    "key": userdata.get("KAGGLE_KEY")
}

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(creds, f)

!chmod 600 ~/.kaggle/kaggle.json

Скачиваем датасет из предыдущей лабораторной работы

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("chicicecream/720p-road-and-traffic-video-for-object-detection")
print("Path to dataset files:", path)
for f in os.listdir(path):
    print(f)

Using Colab cache for faster access to the '720p-road-and-traffic-video-for-object-detection' dataset.
Path to dataset files: /kaggle/input/720p-road-and-traffic-video-for-object-detection
4K Road traffic video for object detection and tracking - free download now.mp4


## Инференс на видео + Мини-проект

Путь к исходному видео

In [ ]:
VIDEO_PATH = "/kaggle/input/720p-road-and-traffic-video-for-object-detection/4K Road traffic video for object detection and tracking - free download now.mp4"

Информация о видео

In [ ]:
import cv2
cap = cv2.VideoCapture(VIDEO_PATH)
print("Открыто:", cap.isOpened())
print("Ширина:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print("Высота:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Кадров всего:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

Открыто: True
Ширина: 1920.0
Высота: 1080.0
FPS: 30.0
Кадров всего: 9184.0


Подсчёт машин и анализ загруженности дороги

In [23]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict

# Пути
MODEL_PATH = "/content/best.pt"
OUTPUT_PATH = "/content/traffic_analysis_output.mp4"

# Параметры
LINE_Y = 400 # y-координата линии подсчёта
DENSITY_THRESHOLDS = {"Free": 5, "Moderate": 15}

# Инициализация
model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (w, h))

# Состояние трекера
track_history = defaultdict(list)    # история центров для треков
counted_ids = set()                  # уже посчитанные ID
total_count = 0                      # общий счётчик

def get_density_label(n):
    if n <= DENSITY_THRESHOLDS["Free"]:
        return "Free", (0, 200, 0)
    elif n <= DENSITY_THRESHOLDS["Moderate"]:
        return "Moderate", (0, 165, 255)
    else:
        return "Traffic jam", (0, 0, 220)


# Основной цикл
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Инференс с трекингом
    results = model.track(frame, persist=True, conf=0.3, iou=0.5, tracker="bytetrack.yaml", verbose=False)

    current_in_frame = 0

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()

        current_in_frame = len(track_ids)

        for box, track_id, conf in zip(boxes, track_ids, confs):
            x1, y1, x2, y2 = map(int, box)
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            # История трека
            track_history[track_id].append((cx, cy))
            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

            # Подсчёт при пересечении линии
            if track_id not in counted_ids:
                prev_points = track_history[track_id]
                if len(prev_points) >= 2:
                    if prev_points[-2][1] < LINE_Y <= prev_points[-1][1] or \
                       prev_points[-2][1] > LINE_Y >= prev_points[-1][1]:
                        counted_ids.add(track_id)
                        total_count += 1

            # Bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 170, 0), 2)
            cv2.putText(frame, f"car #{track_id} {conf:.2f}",
                        (x1, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 170, 0), 1)

            # Хвост трека
            pts = track_history[track_id]
            for i in range(1, len(pts)):
                cv2.line(frame, pts[i-1], pts[i], (255, 170, 0), 1)

    # Линия подсчёта
    cv2.line(frame, (0, LINE_Y), (w, LINE_Y), (0, 255, 255), 2)
    cv2.putText(frame, "calculation line", (10, LINE_Y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # Загруженность
    density_label, density_color = get_density_label(current_in_frame)

    # Информационная панель
    cv2.rectangle(frame, (0, 0), (420, 100), (0, 0, 0), -1)
    cv2.putText(frame, f"Total passed: {total_count}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(frame, f"In frame: {current_in_frame}",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(frame, f"Load: {density_label}",
                (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, density_color, 2)

    out.write(frame)

cap.release()
out.release()

print(f"Видео сохранено в {OUTPUT_PATH}")
print(f"Всего проехало машин через линию: {total_count}")

Видео сохранено в /content/traffic_analysis_output.mp4
Всего проехало машин через линию: 307
